# 🚀 Google Colab Master Runner: Autonomous Rare-Class Skin Lesion Benchmark
### Title: Does Synthetic Data Really Help? A Leakage-Controlled Benchmark of Pretrained Models, Imbalance Losses and Synthetic Augmentation for Minority-Class Skin Lesion Classification

This 100% self-contained Google Colab notebook runs the entire scientific benchmark end to end on free T4 GPU:
1. **GPU & Environment Check** (Tesla T4)
2. **Google Drive Sync** (automatic persistence of checkpoints and logs across disconnections)
3. **Zero-Leakage Stratified Group Splitting** (grouped by `lesion_id`, 5 folds $\times$ 3 seeds = 15 folds)
4. **Execution of All Benchmark Families:**
   - CNN Baselines: `B0` (Plain CE), `B1` (+ Aug), `B2` (+ Balanced Sampler), `B3` (+ Class-Weighted), `B4` (+ Focal Loss)
   - Foundation Models: `P1` (DINOv2 LogReg), `P2` (DINOv2 Weighted MLP), `P3` (BiomedCLIP Linear), `P4` (DINOv2 Fine-Tuned), `P5` (DINOv2 Focal)
   - Generative Augmentation: `S1` (MixUp), `S2` (SD1.5+LoRA @ 50%, 100%, 200%), `FINAL_COMBINED`
5. **Post-Hoc Calibration & Selective Clinical Triage** (Temperature scaling, ECE, 'Uncertain, please review' rule)
6. **Paired Statistical Significance Testing** (Bootstrap CIs & Wilcoxon signed-rank tests)

> **CLINICAL DISCLAIMER:** Research prototype. Not for clinical use or autonomous medical diagnosis.


In [ ]:
# Step 1: Verify Hardware Accelerator
import torch
assert torch.cuda.is_available(), 'GPU not detected! Runtime -> Change runtime type -> T4 GPU.'
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')
print(f'CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# Step 2: Mount Google Drive for Automatic Resumption & Persistence
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/rare_class_benchmark'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive backup path: {DRIVE_DIR}')


In [ ]:
# Step 3: Setup Codebase & Dependencies
import os, sys
if os.path.exists('/content/rare-class-benchmark'):
    %cd /content/rare-class-benchmark
elif os.path.exists('scripts/run_all.py'):
    pass
elif os.path.exists('/content/scripts/run_all.py'):
    %cd /content
else:
    !git clone https://github.com/dhina-research/rare-class-skin-lesion-benchmark.git /content/rare-class-benchmark 2>/dev/null || true
    if os.path.exists('/content/rare-class-benchmark'):
        %cd /content/rare-class-benchmark
!pip install -q -r requirements.txt || pip install -q timm transformers diffusers peft accelerate open_clip_torch scikit-learn scipy pandas matplotlib seaborn pyyaml tqdm


In [ ]:
# Step 4: Download HAM10000 Dataset & Ingest Metadata
from google.colab import files
import shutil
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json file (or skip if downloading via HF mirror):')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
        !chmod 600 /root/.kaggle/kaggle.json

if os.path.exists('/root/.kaggle/kaggle.json'):
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/ --unzip
    !mkdir -p data/images
    !ln -sf data/HAM10000_images_part_1/* data/images/ 2>/dev/null || true
    !ln -sf data/HAM10000_images_part_2/* data/images/ 2>/dev/null || true
else:
    !bash scripts/download_data.sh

# Generate leak-free splits
!python scripts/make_splits.py
!python scripts/prepare_external_set.py


In [ ]:
# Step 5: Interactive Benchmark Arm Selection & Execution
# Select arms to run (all 15 evaluations per arm):
ARMS_TO_RUN = ['B0', 'B1', 'B2', 'B3', 'B4', 'P1', 'P2', 'P3', 'P4', 'P5', 'S1', 'S2_100', 'FINAL_COMBINED']
arms_flag = ' '.join(ARMS_TO_RUN)
!python scripts/run_all.py --arms {arms_flag} --seeds 42 1337 2024 --folds 0 1 2 3 4


In [ ]:
# Step 6: Statistical Analysis & Table Generation
from src.stats import aggregate_benchmark_runs, compute_paired_comparison
df_summary, df_runs = aggregate_benchmark_runs('results/logs/runs.csv', 'results/tables')
display(df_summary[['arm_id', 'n_evals', 'test_minority_recall_formatted', 'test_balanced_accuracy_formatted', 'test_macro_f1_formatted']])

# Sync to Google Drive
!cp -r results/logs/runs.csv {DRIVE_DIR}/ 2>/dev/null || true
!cp -r results/tables/ {DRIVE_DIR}/ 2>/dev/null || true
!cp -r results/figures/ {DRIVE_DIR}/ 2>/dev/null || true

# Download complete bundle
!zip -r /content/complete_benchmark_results.zip results/
files.download('/content/complete_benchmark_results.zip')
